<a href="https://colab.research.google.com/github/nmit-1NT23CS128/Social-Media-Comment-Moderation/blob/main/Social_Media_Comment_Moderation_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sympy==1.14.0 --force-reinstall
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 49.9 MB/s eta 0:00:00
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0


In [ ]:
!pip install transformers datasets torch scikit-learn pandas numpy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#from huggingface_hub import notebook_login
#notebook_login()


In [7]:
import pandas as pd
import os

# ==============================
# MOUNT GOOGLE DRIVE (COLAB)
# ==============================
from google.colab import drive
drive.mount('/content/drive')

# ==============================
# FUNCTION TO PROCESS DATAFRAME
# ==============================
def process_df(df):

    # Possible text column names
    possible_text_cols = [
        'comment_text',
        'Comment',
        'Text',
        'Sentence',
        'text_content',
        'Tweet',
        'comment',
        'text',
        'content',
        'message',
        'review',
        'body'
    ]

    found = False

    # Rename matching column to comment_text
    for col in possible_text_cols:
        if col in df.columns:
            df = df.rename(columns={col: 'comment_text'})
            found = True
            print(f"Using text column: {col}")
            break

    # If no text column found
    if not found:
        raise ValueError(
            f"No suitable text column found.\nColumns available: {df.columns.tolist()}"
        )

    # Convert comment_text to string
    df['comment_text'] = df['comment_text'].fillna('').astype(str)

    # Toxicity columns
    toxicity_columns = [
        'toxic',
        'severe_toxic',
        'obscene',
        'threat',
        'insult',
        'identity_hate'
    ]

    # Ensure toxicity columns exist
    for col in toxicity_columns:

        if col not in df.columns:
            df[col] = 0

        else:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].fillna(0).astype(int)
            df[col] = df[col].apply(lambda x: 1 if x > 0 else 0)

    return df


# ==============================
# BASE PATH
# ==============================
base_toxic_data_path = "/content/drive/MyDrive/toxic_comment/"

# ==============================
# FILE LISTS
# ==============================
additional_csv_files = [
    "Toxicity_detection_context.csv",
    "toxic_comments_dataset.csv",
    "YouTube Comments Dataset with Sentiment Toxicity and Spam Labels (45K Rows).csv"
]

additional_xlsx_files = [
    "Toxic Comment Moderation Dataset.xlsx"
]

# ==============================
# STORE ALL DATAFRAMES
# ==============================
all_dfs = []

# ==============================
# LOAD ORIGINAL TRAIN.CSV
# ==============================
train_path = "/content/drive/MyDrive/train.csv"

print("\nChecking original train.csv...")

if os.path.exists(train_path):

    try:
        df_original_train = pd.read_csv(train_path)

        print("Columns:")
        print(df_original_train.columns.tolist())

        processed_df = process_df(df_original_train)

        all_dfs.append(processed_df)

        print("Successfully loaded train.csv")

    except Exception as e:
        print(f"Error processing train.csv: {e}")

else:
    print("train.csv NOT FOUND")


# ==============================
# LOAD CSV FILES
# ==============================
for file_name in additional_csv_files:

    full_path = os.path.join(base_toxic_data_path, file_name)

    print(f"\nChecking {file_name}...")

    if os.path.exists(full_path):

        try:
            df_additional = pd.read_csv(full_path)

            print("Columns:")
            print(df_additional.columns.tolist())

            processed_df = process_df(df_additional)

            all_dfs.append(processed_df)

            print(f"Successfully loaded {file_name}")

        except Exception as e:
            print(f"Error processing {file_name}: {e}")

    else:
        print(f"File NOT FOUND: {full_path}")


# ==============================
# LOAD XLSX FILES
# ==============================
for file_name in additional_xlsx_files:

    full_path = os.path.join(base_toxic_data_path, file_name)

    print(f"\nChecking {file_name}...")

    if os.path.exists(full_path):

        try:
            df_additional = pd.read_excel(full_path)

            print("Columns:")
            print(df_additional.columns.tolist())

            processed_df = process_df(df_additional)

            all_dfs.append(processed_df)

            print(f"Successfully loaded {file_name}")

        except Exception as e:
            print(f"Error processing {file_name}: {e}")

    else:
        print(f"File NOT FOUND: {full_path}")


# ==============================
# COMBINE ALL DATAFRAMES
# ==============================
print("\n==============================")
print("TOTAL DATAFRAMES LOADED:", len(all_dfs))
print("==============================")

if len(all_dfs) > 0:

    df = pd.concat(all_dfs, ignore_index=True)

    # Convert alternative labels into toxic column
    if 'label' in df.columns:
        df['toxic'] = df['toxic'] | (
            df['label'].astype(str).str.lower().isin(['toxic', '1', 'yes'])
        ).astype(int)

    if 'label_toxicity' in df.columns:
        df['toxic'] = df['toxic'] | (
            pd.to_numeric(df['label_toxicity'], errors='coerce')
            .fillna(0)
            .astype(int)
        )

    if 'is_toxic' in df.columns:
        df['toxic'] = df['toxic'] | (
            pd.to_numeric(df['is_toxic'], errors='coerce')
            .fillna(0)
            .astype(int)
        )

    # Define required columns
    required_columns = [
        'comment_text',
        'toxic',
        'severe_toxic',
        'obscene',
        'threat',
        'insult',
        'identity_hate'
    ]

    # Create missing columns if needed (they should already exist from process_df, but as a safeguard)
    for col in required_columns:
        if col not in df.columns:
            df[col] = 0

    # Select only required columns
    df = df[required_columns]

    # Fill NaN values (should be minimal after process_df and label conversion)
    df = df.fillna(0)

    # Remove duplicates
    df.drop_duplicates(subset=['comment_text'], inplace=True)

    print("\nCombined DataFrame Shape (after cleanup):")
    print(df.shape)

    print("\nSample Data (after cleanup):")
    display(df.head())

else:
    print("\nNo dataframes were loaded.")
    print("Check:")
    print("1. Google Drive mounted correctly")
    print("2. File paths are correct")
    print("3. File names are correct")
    print("4. Dataset contains text columns")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Checking original train.csv...
Columns:
['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
Using text column: comment_text
Successfully loaded train.csv

Checking Toxicity_detection_context.csv...
Columns:
['text', ' label']
Using text column: text
Successfully loaded Toxicity_detection_context.csv

Checking toxic_comments_dataset.csv...
Columns:
['comment_id', 'user_name', 'platform', 'comment_text', 'label', 'language', 'timestamp', 'length']
Using text column: comment_text
Successfully loaded toxic_comments_dataset.csv

Checking YouTube Comments Dataset with Sentiment Toxicity and Spam Labels (45K Rows).csv...
Columns:
['channel_username', 'channel_id', 'video_id', 'video_title', 'comment_text', 'comment_id', 'author', 'published_at', 'like_count', 'reply_count', 'language', 'label_sentiment', 'label_toxicity',

,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [8]:
toxicity_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

df['label'] = df[toxicity_columns].max(axis=1)


In [10]:
texts = df['comment_text'].fillna("")
labels = df['label']


In [11]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)


In [12]:
print(train_labels.value_counts())


label
0    171976
1     14983
Name: count, dtype: int64


In [13]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
import torch


In [15]:
class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)


In [16]:
train_dataset = ToxicDataset(train_texts, train_labels, tokenizer)
test_dataset = ToxicDataset(test_texts, test_labels, tokenizer)

In [17]:
print(len(train_dataset))
print(len(test_dataset))


186959
46740


In [19]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',

    num_train_epochs=1,

    per_device_train_batch_size=4,   # reduced
    per_device_eval_batch_size=4,    # reduced

    gradient_accumulation_steps=4,   # acts like batch size 16

    learning_rate=2e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    fp16=True   # ⭐ THIS IS STEP 5 (add this line)
)


In [24]:
from transformers import Trainer


In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)


In [26]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("/content/drive/MyDrive/toxic_model")
tokenizer.save_pretrained("/content/drive/MyDrive/toxic_model")

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)

accuracy = accuracy_score(test_labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average='binary')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
# training_args = TrainingArguments(
#     output_dir='./results',
#     num_train_epochs=3,          # increase epochs
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     learning_rate=2e-5,          # KEY CHANGE
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     weight_decay=0.01            # regularization
# )


In [ ]:
# from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# # Re-initialize tokenizer if it's not defined (due to potential kernel restart)
# tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# # Re-initialize model if it's not defined (due to potential kernel restart)
# model = DistilBertForSequenceClassification.from_pretrained(
#     'distilbert-base-uncased',
#     num_labels=2
# )
# model.save_pretrained("/content/drive/MyDrive/toxic_model")
# tokenizer.save_pretrained("/content/drive/MyDrive/toxic_model")

In [ ]:
!mkdir BERT_Toxic_Project

In [ ]:
!ls

In [ ]:
%cd BERT_Toxic_Project

In [ ]:
pip install matplotlib scikit-learn

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import torch
import torch.nn.functional as F
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix



# Page config + styling

st.set_page_config(page_title="Toxic Comment Detector", layout="centered")

st.markdown("""

<style>
.main {
    background-color: #0e1117;
}
.stTextArea textarea {
    font-size: 16px;
}
</style>

""", unsafe_allow_html=True)

# Load model

model = DistilBertForSequenceClassification.from_pretrained(
"/content/drive/MyDrive/toxic_model"
)

tokenizer = DistilBertTokenizer.from_pretrained(
"/content/drive/MyDrive/toxic_model"
)

model.eval()

# Sidebar navigation

page = st.sidebar.selectbox(
"Select Page",
["Toxic Comment Detector", "Model Performance"]
)

# -------------------------------

# PAGE 1: LIVE PREDICTION

# -------------------------------

if page == "Toxic Comment Detector":
    st.markdown("## 🛡 AI Toxic Comment Moderator")
    st.caption("Detects and filters harmful comments in real-time")

    def normalize_text(text):
        text = text.lower()
        text = text.replace("ur", "you are")
        text = text.replace("u ", "you ")
        return text


    user_input = st.text_area("Enter Comment", key="input_text")

    if user_input.strip() == "":
        st.info("Please enter a comment")
    else:
        if st.button("Predict"):

            processed = normalize_text(user_input)



            inputs = tokenizer(
                processed,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=128
            )

            # Spinner added here
            with st.spinner("Analyzing comment..."):
                with torch.no_grad():
                    outputs = model(**inputs)
                    probs = F.softmax(outputs.logits, dim=1)
                    toxic_prob = probs[0][1].item()

            if toxic_prob >0.85:
              st.error("Highly Toxic -🚨 Comment Blocked")
              decision = "Automatically Hidden"

            elif toxic_prob >0.4:
                st.warning("Moderately Toxic - ⚠ Comment Flagged")
                decision = "Blocked (Pending Risk)"

            else:
                st.success("Less Toxic - ✅ Comment Approved")
                decision = "Published"

            # Display probability
            st.write("Toxic Probability:", round(toxic_prob, 3))
            st.progress(int(toxic_prob * 100))

            st.caption(f"Confidence Score: {round(toxic_prob, 3)}")
            st.progress(int(toxic_prob * 100))

            # System Action
            st.write(f"🛠 System Action: **{decision}**")

# -------------------------------

# PAGE 2: MODEL DASHBOARD

# -------------------------------

else:

    st.title("📊 Model Performance Dashboard")

    accuracy = 0.951
    precision = 0.763
    recall = 0.754
    f1 = 0.758

    st.metric("Accuracy", accuracy)
    st.metric("Precision", precision)
    st.metric("Recall", recall)
    st.metric("F1 Score", f1)

    # Example confusion matrix
    y_true = [0, 0, 0, 1, 1, 1]
    y_pred = [0, 0, 1, 1, 1, 0]

    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots()
    ax.matshow(cm)

    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, val, ha='center', va='center')

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    st.pyplot(fig)

In [ ]:
!ls

In [ ]:
model.save_pretrained("/content/drive/MyDrive/toxic_model")
tokenizer.save_pretrained("/content/drive/MyDrive/toxic_model")

In [ ]:
!pip install pyngrok

In [ ]:
!pip install pyngrok streamlit

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3BDTF8NoUigVbuJIvmmnidQzfQm_4UTY1Ysq7tMr8ihd3iAC9")

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

In [ ]:
!ls /content/drive/MyDrive/toxic_model